<div style='background:#1a3a5c;color:white;padding:22px 30px;border-radius:10px;font-family:sans-serif'>
<h1 style='margin:0 0 6px 0;font-size:1.6em'>⚡ DESAFÍO RELÁMPAGO — Sesión 05 · SOLUCIONES (Profesor)</h1>
<h2 style='margin:0 0 10px 0;font-weight:300;font-size:1.1em'>La STFT: Ver y Manipular el Sonido</h2>
<p style='margin:0;opacity:0.8;font-size:0.95em'>
Esta versión incluye: respuestas a las preguntas de predicción, código completo,
explicaciones pedagógicas detalladas y notas de facilitación para clase.
</p></div>

## 0 · Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft as scipy_stft, istft as scipy_istft
import wave, struct, os

FS = 44100

def save_wav(filename, audio, fs=FS):
    """[WHY] int16 PCM — único formato garantizado en macOS/afplay."""
    audio = np.array(audio, dtype=np.float64)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak * 0.88
    pcm16 = np.clip(audio * 32767, -32768, 32767).astype(np.int16)
    with wave.open(filename, 'w') as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(fs)
        wf.writeframes(pcm16.tobytes())
    print(f'  Guardado: {filename}  ({os.path.getsize(filename)/1024:.0f} KB)')

print('Setup OK')

## 1 · Predicciones — Respuestas Esperadas

| Señal | Descripción | Respuesta correcta |
|-------|-------------|--------------------|
| **A** | Tono puro 440 Hz, 2 s | Línea horizontal fina y brillante a 440 Hz, constante en el tiempo |
| **B** | Armónicos 110–550 Hz | 5 líneas horizontales equidistantes (110, 220, 330, 440, 550 Hz) — la más brillante suele ser la más baja |
| **C** | Sweep 200→3000 Hz | Línea diagonal ascendente de izquierda a derecha |

**Nota de facilitación:** Los errores más frecuentes son creer que el sweep produce una nube (confunden con ruido) y no anticipar el múltiple de armónicos. Si el grupo predice bien la señal C pero no la B, reforzar que armónico = múltiplo exacto = líneas equidistantes en frecuencia.

In [ ]:
dur = 2.0
N = int(dur * FS)
t = np.linspace(0, dur, N, endpoint=False)

sig_A = np.sin(2*np.pi*440*t) * 0.7
sig_B = sum(np.sin(2*np.pi*110*k*t) / k for k in range(1, 6)) * 0.4
phase_sweep = 2*np.pi * np.cumsum(np.linspace(200, 3000, N)) / FS
sig_C = np.sin(phase_sweep) * 0.7

for name, sig in [('dr05_A_tono440', sig_A), ('dr05_B_armonicos', sig_B), ('dr05_C_sweep', sig_C)]:
    save_wav(f'{name}.wav', sig)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, (title, sig) in zip(axes, [
    ('Señal A — Tono puro 440 Hz', sig_A),
    ('Señal B — Armónicos de 110 Hz', sig_B),
    ('Señal C — Sweep 200→3000 Hz', sig_C),
]):
    ax.specgram(sig, NFFT=2048, Fs=FS, noverlap=1024, cmap='magma')
    ax.set_ylim(0, 4000)
    ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('Frecuencia (Hz)'); ax.set_title(title)
plt.tight_layout()
plt.show()

## 2 · Trade-off — Respuesta Esperada

- **N=256 (corto):** La trayectoria del sweep es nítida (buena resolución temporal), pero cada línea es gruesa en frecuencia (mala resolución frecuencial).
- **N=4096 (largo):** La frecuencia exacta se ve como una línea muy fina, pero la trayectoria del sweep es borrosa (mala resolución temporal).
- **N=1024 (medio):** Balance razonable.

**Conclusión:** No se puede tener buena resolución en tiempo Y en frecuencia simultáneamente. Esto es el **principio de incertidumbre de Heisenberg** aplicado al DSP: $\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$.

**Nota de facilitación:** La pregunta del Devil's Advocate típica es "¿por qué no usamos una ventana adaptativa?". La respuesta es que existen (STFT adaptativa, wavelet transform), pero tienen sus propias limitaciones. El trade-off es fundamental, no una limitación de implementación.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, n_fft in zip(axes, [256, 1024, 4096]):
    hop = n_fft // 4
    ax.specgram(sig_C, NFFT=n_fft, Fs=FS, noverlap=n_fft - hop, cmap='magma')
    ax.set_ylim(0, 4000)
    ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('Frecuencia (Hz)')
    ax.set_title(f'N_FFT = {n_fft}\n(ventana = {n_fft/FS*1000:.0f} ms)')
plt.suptitle('Trade-off: Resolución temporal vs. frecuencial', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3 · STFT Compleja — Solución Completa

El punto clave es que `np.fft.rfft` devuelve números complejos — no hay que hacer nada especial para conservar la fase, solo **no tomar el módulo**.

In [ ]:
def my_stft(x, n_fft=2048, hop=512):
    """
    STFT compleja — retorna X de shape (n_fft//2+1, n_frames), dtype=complex.
    [WHY] rfft devuelve valores complejos que contienen magnitud Y fase.
    """
    window = np.hanning(n_fft)
    # Zero-pad para que el último frame cubra toda la señal
    remainder = (len(x) - n_fft) % hop
    if remainder != 0:
        pad_len = hop - remainder
        x = np.concatenate([x, np.zeros(pad_len)])
    n_frames = (len(x) - n_fft) // hop + 1
    X = np.zeros((n_fft // 2 + 1, n_frames), dtype=complex)
    for m in range(n_frames):
        frame = x[m * hop : m * hop + n_fft]
        X[:, m] = np.fft.rfft(frame * window)   # ← SOLUCIÓN: rfft sin tomar abs
    return X

def my_istft(X, n_fft=2048, hop=512, signal_len=None):
    """
    ISTFT por Overlap-Add.
    [WHY] Hann con 50% overlap cumple COLA → reconstrucción perfecta.
    """
    n_frames = X.shape[1]
    output_len = signal_len or (n_frames - 1) * hop + n_fft
    y = np.zeros(output_len)
    window = np.hanning(n_fft)
    for m in range(n_frames):
        frame = np.fft.irfft(X[:, m], n=n_fft)
        start = m * hop
        end = min(start + n_fft, output_len)
        y[start:end] += (frame * window)[:end - start]
    norm = np.zeros(output_len)
    win_sq = window ** 2
    for m in range(n_frames):
        start = m * hop
        end = min(start + n_fft, output_len)
        norm[start:end] += win_sq[:end - start]
    norm = np.where(norm > 1e-8, norm, 1.0)
    return y / norm

# Verificación
n_fft, hop = 2048, 512
X = my_stft(sig_A, n_fft=n_fft, hop=hop)
rec = my_istft(X, n_fft=n_fft, hop=hop, signal_len=len(sig_A))
# Comparar solo la zona central bien normalizada (los bordes tienen efecto de edge)
margin = n_fft
error = np.max(np.abs(sig_A[margin:-margin] - rec[margin:-margin]))
print(f'Error máximo de reconstrucción (zona central): {error:.2e}  (< 1e-10 es perfecto con Hann)')
assert error < 1e-4, 'Error demasiado alto — revisar implementación'
save_wav('dr05_verificacion_reconstruccion.wav', rec)
print('✓ STFT + ISTFT OK')

## 4 · La Fase — Respuestas Esperadas

**Predicciones sobre qué pasa al poner la fase a cero en el tono 440 Hz:**

1. **¿Sigue sonando como 440 Hz?** Sí — la frecuencia del robot depende del hop, no de la frecuencia original. La magnitud (que sí incluye el pico a 440 Hz) se conserva, así que el tono de 440 Hz sigue presente.
2. **¿Cambia el timbre?** Sí, notablemente. La fase cero alinea todos los cosenos de todos los bins al inicio de cada frame, creando un sonido más duro y periódico — más "metálico".
3. **¿Escuchas clics?** Depende del hop. Con Hann y 50% overlap no hay clics porque COLA está satisfecha. Con hop=N (0% overlap) sí habrá clics porque los frames no se solapan.

**Nota de facilitación:** La imagen de fase (el segundo subplot) es la más sorprendente para el grupo — parece ruido aleatorio. Esto es normal: la fase varía rápidamente con el tiempo y con la frecuencia, incluso para un tono puro. No confundir "fase aparentemente aleatoria" con "fase sin información".

In [ ]:
f_bins, t_frames, X_A = scipy_stft(sig_A, fs=FS, nperseg=2048, noverlap=2048-512,
                                    window='hann', boundary='zeros')
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

im = axes[0,0].pcolormesh(t_frames, f_bins[:200], 20*np.log10(np.abs(X_A[:200])+1e-8),
                          cmap='magma', shading='auto')
axes[0,0].set_title('Magnitud |X(m,k)| en dB'); axes[0,0].set_ylabel('Frecuencia (Hz)'); axes[0,0].set_xlabel('Tiempo (s)')
plt.colorbar(im, ax=axes[0,0], label='dB')

im2 = axes[0,1].pcolormesh(t_frames, f_bins[:200], np.angle(X_A[:200]),
                            cmap='hsv', shading='auto', vmin=-np.pi, vmax=np.pi)
axes[0,1].set_title('Fase ∠X(m,k) — aparece como ruido (es normal)'); axes[0,1].set_ylabel('Frecuencia (Hz)'); axes[0,1].set_xlabel('Tiempo (s)')
plt.colorbar(im2, ax=axes[0,1], label='rad')

axes[1,0].plot(np.arange(len(sig_A))/FS, sig_A, color='steelblue', lw=0.5)
axes[1,0].set_title('Señal original'); axes[1,0].set_xlabel('Tiempo (s)'); axes[1,0].set_ylabel('Amplitud')

X_zerophase = np.abs(X_A) * np.exp(1j * 0)
_, sig_robot_preview = scipy_istft(X_zerophase, fs=FS, nperseg=2048, noverlap=2048-512,
                                   window='hann', boundary='zeros')
l = min(len(sig_A), len(sig_robot_preview))
axes[1,1].plot(np.arange(l)/FS, sig_robot_preview[:l], color='crimson', lw=0.5)
axes[1,1].set_title('Fase cero — mismo espectro, distinto sonido'); axes[1,1].set_xlabel('Tiempo (s)'); axes[1,1].set_ylabel('Amplitud')

plt.tight_layout(); plt.show()
save_wav('dr05_tono_fase_cero.wav', sig_robot_preview)

## 5 · Robotize y Whisperize — Solución Completa

**Punto clave para el grupo:** `np.exp(1j * 0) = 1`, así que `|X| * exp(0j) = |X|` — todos los valores quedan reales positivos. Esto equivale a alinear todos los cosenos al inicio de cada frame.

**Conexión con física:** La frecuencia del robot es $f_s/\text{hop}$. Para que el robot "cante" en La3 (220 Hz): `hop = 44100 / 220 = 200.5 ≈ 200 muestras`.

In [ ]:
def robotize(x, n_fft=2048, hop=512):
    """
    [WHY] Fase cero = todos los cosenos alineados al inicio de cada frame.
    Periodicidad artificial a tasa fs/hop → pitch del robot.
    """
    _, _, X = scipy_stft(x, fs=FS, nperseg=n_fft, noverlap=n_fft-hop,
                         window='hann', boundary='zeros')
    X_robot = np.abs(X) * np.exp(1j * 0)   # ← fase cero, magnitud conservada
    _, y = scipy_istft(X_robot, fs=FS, nperseg=n_fft, noverlap=n_fft-hop,
                       window='hann', boundary='zeros')
    return y

def whisperize(x, n_fft=2048, hop=512):
    """
    [WHY] Fase aleatoria destruye coherencia temporal → desaparece el pitch.
    La magnitud (formantes) se conserva → inteligibilidad parcial del habla.
    """
    _, _, X = scipy_stft(x, fs=FS, nperseg=n_fft, noverlap=n_fft-hop,
                         window='hann', boundary='zeros')
    random_phase = np.random.uniform(0, 2*np.pi, X.shape)  # ← uniforme [0, 2π]
    X_whisper = np.abs(X) * np.exp(1j * random_phase)
    _, y = scipy_istft(X_whisper, fs=FS, nperseg=n_fft, noverlap=n_fft-hop,
                       window='hann', boundary='zeros')
    return y

# Demostración con hop=512 vs hop=200 (≈220 Hz)
print('Generando variantes...')
for hop, label in [(512, 'hop512'), (200, 'hop200_La3')]:
    f_robot = FS / hop
    print(f'  hop={hop} → robot canta a {f_robot:.0f} Hz')
    robot = robotize(sig_B, hop=hop)   # sig_B tiene más riqueza armónica
    whisper = whisperize(sig_B, hop=hop)
    save_wav(f'dr05_robot_{label}.wav', robot)
    save_wav(f'dr05_whisper_{label}.wav', whisper)

print('\nFrecuencia robot = fs / hop')
print('Para que cante en una nota específica: hop = fs / f_nota')

In [ ]:
# Comparación visual: espectrogramas antes/después de robotización
robot_sig = robotize(sig_B, hop=512)
whisper_sig = whisperize(sig_B, hop=512)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, sig) in zip(axes, [
    ('Original — Armónicos 110 Hz', sig_B),
    ('Robot (fase=0, hop=512)', robot_sig[:len(sig_B)]),
    ('Whisper (fase aleatoria)', whisper_sig[:len(sig_B)]),
]):
    ax.specgram(sig, NFFT=2048, Fs=FS, noverlap=1024, cmap='magma')
    ax.set_ylim(0, 4000)
    ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('Frecuencia (Hz)'); ax.set_title(title)

plt.suptitle('Magnitud idéntica en los tres — la diferencia es solo la FASE', y=1.02)
plt.tight_layout()
plt.show()
print('CLAVE PEDAGÓGICA: los tres espectrogramas se ven casi idénticos.')
print('El cambio de fase no se ve en el espectrograma de magnitud — pero SÍ se escucha.')

## Notas de Facilitación

### Errores frecuentes de los estudiantes

1. **Confundir `np.angle(X)` con ruido aleatorio.** La fase de una señal real tiene estructura — varía de forma determinista — pero esa estructura es difícil de ver sin herramientas especiales. Es normal que parezca ruido.

2. **Sumar ruido a la fase en vez de reemplazarla.** El tip de la fun_task lo menciona: si `X_whisper = |X| * exp(j*(angle(X) + random))`, el efecto no es tan pronunciado como si se reemplaza la fase completamente: `X_whisper = |X| * exp(j*random)`.

3. **Olvidar la condición COLA.** Si usan `hop = n_fft` (sin overlap), la reconstrucción tiene clics en cada frontera de frame. La prueba de reconstrucción (STFT→ISTFT sin modificar) es el diagnóstico correcto.

4. **Usar el wav con float32 (el bug clásico).** Con `setsampwidth(4)` y datos float, afplay produce ruido. Siempre int16.

### Pregunta de cierre — Respuesta esperada

"¿Qué parte del habla está codificada en la magnitud (y no en la fase)?"

Respuesta: Los **formantes** — las resonancias del tracto vocal que determinan las vocales. Por eso el susurro es inteligible: las vocales (a, e, i, o, u) sobreviven porque dependen de la magnitud del espectro. Los **fonemas consonánticos** con pitch (como 'v', 'z') pierden su tono pero mantienen su posición espectral.